# Preselección de variables 

Esta celda importa las herramientas necesarias para la preselección,o separa los predictores `X` del target `compra` y comprueba dimensiones, balance, NaN y nombres.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path.cwd().parent
state_path = PROJECT_ROOT / '.github' / 'copilot-instructions.md'
state_text = state_path.read_text(encoding='utf-8')
state_section = state_text.split('## ESTADO ACTUAL DEL PROYECTO', 1)[1]
input_path_text = state_section.split('`', 2)[1]
input_path = (Path.cwd() / input_path_text).resolve()
df = pd.read_pickle(input_path)
TARGET = 'compra'
X = df.drop(columns=[TARGET])
y = df[TARGET]
print(f'Input path from copilot instructions: {input_path}')
print(f'Dataframe: {df.shape}; X: {X.shape}; target: {TARGET}')
print(f'Target distribution: {y.value_counts(normalize=True).sort_index().round(4).to_dict()}')
print(f'Unexpected NaN in X: {int(X.isna().sum().sum())}')
print(f'Unique columns: {bool(X.columns.is_unique)}')
df.info()

Input path from copilot instructions: C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\02_datos\03_Entrenamiento\04_train_tablon_transformado.pkl
Dataframe: (6360, 53); X: (6360, 52); target: compra
Target distribution: {0: 0.6252, 1: 0.3748}
Unexpected NaN in X: 0
Unique columns: True
<class 'pandas.DataFrame'>
Index: 6360 entries, 2954 to 7270
Data columns (total 53 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   compra                                 6360 non-null   int64  
 1   visitas_total_yj_mm                    6360 non-null   float64
 2   tiempo_en_site_total_yj_mm             6360 non-null   float64
 3   paginas_vistas_visita_yj_mm            6360 non-null   float64
 4   score_actividad_mm                     6360 non-null   float64
 5   score_perfil_mm                        6360 non-null   float64
 6   score_actividad_missing                6360 non-null   float64
 7   o

Se resume cada predictor con su tipo, cantidad de valores únicos, faltantes y varianza.
La tabla permite detectar columnas constantes o con poca variación antes de seleccionar.
También deja explícito que el problema es clasificación y que el modelo objetivo es lineal.

In [2]:
diagnostico = pd.DataFrame({
    'variable': X.columns,
    'dtype': X.dtypes.astype(str).values,
    'unicos': X.nunique(dropna=True).values,
    'missing': X.isna().sum().values,
    'varianza': X.var(numeric_only=True).reindex(X.columns).values,
})
display(diagnostico)
print('Modelo objetivo: regresión logística (familia lineal).')
print('Recomendación de la skill: aplicar preselección supervisada porque los modelos lineales son sensibles a variables irrelevantes y multicolinealidad.')

,variable,dtype,unicos,missing,varianza
0,visitas_total_yj_mm,float64,31,0,0.042049
1,tiempo_en_site_total_yj_mm,float64,1570,0,0.102026
2,paginas_vistas_visita_yj_mm,float64,92,0,0.039775
3,score_actividad_mm,float64,12,0,0.008943
4,score_perfil_mm,float64,10,0,0.022296
5,score_actividad_missing,float64,2,0,0.248406
6,origen_Landing Page Submission,float64,2,0,0.248765
7,origen_Lead Add Form,float64,2,0,0.058534
8,origen_Lead Import,float64,2,0,0.005940
9,fuente_Direct Traffic,float64,2,0,0.201853


Modelo objetivo: regresión logística (familia lineal).
Recomendación de la skill: aplicar preselección supervisada porque los modelos lineales son sensibles a variables irrelevantes y multicolinealidad.


## Bloque 1 — método supervisado

Se ejecuta RFECV con regresión logística penalizada L1 y cinco particiones estratificadas.
El método prueba distintos tamaños de conjunto y conserva la cantidad que obtiene mejor ROC AUC medio.
Finalmente muestra las variables seleccionadas, sus importancias y las que quedaron fuera para revisión.

In [3]:
# RFECV estándar — revisión 1
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
l1_estimator = LogisticRegression(penalty='l1', solver='saga', max_iter=5000, random_state=42)
rfecv = RFECV(
    estimator=l1_estimator,
    step=1,
    min_features_to_select=1,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
)
rfecv.fit(X, y)
variables_supervisadas = X.columns[rfecv.support_].tolist()
variables_excluidas_rfecv = X.columns[~rfecv.support_].tolist()
importancias_supervisadas = pd.Series(np.abs(rfecv.estimator_.coef_[0]), index=variables_supervisadas).sort_values(ascending=False)
print(f'RFECV óptimo: {rfecv.n_features_} de {X.shape[1]} variables.')
print(f'Mejor ROC AUC medio CV: {rfecv.cv_results_["mean_test_score"][rfecv.n_features_-1]:.4f}')
print('Variables seleccionadas:')
display(pd.DataFrame({'variable': variables_supervisadas, 'importancia_abs_L1': importancias_supervisadas.reindex(variables_supervisadas).values}).sort_values('importancia_abs_L1', ascending=False))
print('Variables excluidas por RFECV:')
display(pd.DataFrame({'variable': variables_excluidas_rfecv}))

RFECV óptimo: 43 de 52 variables.
Mejor ROC AUC medio CV: 0.8976
Variables seleccionadas:


,variable,importancia_abs_L1
3,score_actividad_mm,7.906052
1,tiempo_en_site_total_yj_mm,4.761277
7,origen_Lead Add Form,3.106766
20,ult_actividad_SMS Sent,3.100028
42,visitas_total_missing,2.629270
2,paginas_vistas_visita_yj_mm,2.103219
18,ult_actividad_Otros,2.095194
4,score_perfil_mm,2.072426
9,fuente_Direct Traffic,2.057756
17,ult_actividad_Email Opened,1.879565


Variables excluidas por RFECV:


,variable
0,no_llamar_Yes
1,ambito_Finance Management
2,ambito_Marketing Management
3,ambito_Media and Advertising
4,ambito_Travel and Tourism
5,ocupacion_Other
6,conociste_google_Yes
7,conociste_facebook_Yes
8,conociste_referencias_Yes


## Bloque 2 — revisión por variable madre

Esta celda agrupa las variables derivadas según la variable original de la matriz.
Las dummies One-Hot y los indicadores se conservan como grupo, sin mezclar sus representaciones.
Las ramas numéricas alternativas se compararían por importancia; aquí cada madre tiene una única rama final.

In [4]:
# Deduplicación por variable madre — revisión 2
from collections import OrderedDict

all_groups = OrderedDict([
    ('visitas_total', ['visitas_total_yj_mm', 'visitas_total_missing']),
    ('tiempo_en_site_total', ['tiempo_en_site_total_yj_mm']),
    ('paginas_vistas_visita', ['paginas_vistas_visita_yj_mm']),
    ('score_actividad', ['score_actividad_mm', 'score_actividad_missing']),
    ('score_perfil', ['score_perfil_mm']),
    ('origen', [c for c in X.columns if c.startswith('origen_')]),
    ('fuente', [c for c in X.columns if c.startswith('fuente_')]),
    ('no_enviar_email', [c for c in X.columns if c.startswith('no_enviar_email_')]),
    ('no_llamar', [c for c in X.columns if c.startswith('no_llamar_')]),
    ('ult_actividad', [c for c in X.columns if c.startswith('ult_actividad_')]),
    ('ambito', [c for c in X.columns if c.startswith('ambito_')]),
    ('ocupacion', [c for c in X.columns if c.startswith('ocupacion_')]),
    ('conociste_google', [c for c in X.columns if c.startswith('conociste_google_')]),
    ('conociste_facebook', [c for c in X.columns if c.startswith('conociste_facebook_')]),
    ('conociste_referencias', [c for c in X.columns if c.startswith('conociste_referencias_')]),
    ('descarga_lm', [c for c in X.columns if c.startswith('descarga_lm_')]),
])
rows=[]
variables_tras_depuracion_por_madre=[]
for mother, derivatives in all_groups.items():
    selected=[c for c in derivatives if c in variables_supervisadas]
    removed=[c for c in derivatives if c not in variables_supervisadas]
    if selected:
        variables_tras_depuracion_por_madre.extend(selected)
    rows.append({'madre':mother,'tipo_grupo':'OHE/flags' if any(c.startswith(('origen_','fuente_','no_','ult_','ambito_','ocupacion_','conociste_','descarga_')) for c in derivatives) else 'numérica + flags','conservadas':', '.join(selected) or 'ninguna','fuera_por_RFECV':', '.join(removed) or 'ninguna','acción_deduplicación':'mantener grupo seleccionado'})
dedup_decisions=pd.DataFrame(rows)
display(dedup_decisions[dedup_decisions['conservadas']!='ninguna'])
print(f'Variables después de RFECV: {len(variables_supervisadas)}')
print(f'Variables después de deduplicación por madre: {len(variables_tras_depuracion_por_madre)}')
print('No se eliminaron variables adicionales por deduplicación: no hay ramas numéricas alternativas ni codificaciones duplicadas.')

,madre,tipo_grupo,conservadas,fuera_por_RFECV,acción_deduplicación
0,visitas_total,numérica + flags,"visitas_total_yj_mm, visitas_total_missing",ninguna,mantener grupo seleccionado
1,tiempo_en_site_total,numérica + flags,tiempo_en_site_total_yj_mm,ninguna,mantener grupo seleccionado
2,paginas_vistas_visita,numérica + flags,paginas_vistas_visita_yj_mm,ninguna,mantener grupo seleccionado
3,score_actividad,numérica + flags,"score_actividad_mm, score_actividad_missing",ninguna,mantener grupo seleccionado
4,score_perfil,numérica + flags,score_perfil_mm,ninguna,mantener grupo seleccionado
5,origen,OHE/flags,"origen_Landing Page Submission, origen_Lead Ad...",ninguna,mantener grupo seleccionado
6,fuente,OHE/flags,"fuente_Direct Traffic, fuente_Google, fuente_O...",ninguna,mantener grupo seleccionado
7,no_enviar_email,OHE/flags,no_enviar_email_Yes,ninguna,mantener grupo seleccionado
9,ult_actividad,OHE/flags,"ult_actividad_Converted to Lead, ult_actividad...",ninguna,mantener grupo seleccionado
10,ambito,OHE/flags,"ambito_Business Administration, ambito_Descono...","ambito_Finance Management, ambito_Marketing Ma...",mantener grupo seleccionado


Variables después de RFECV: 43
Variables después de deduplicación por madre: 43
No se eliminaron variables adicionales por deduplicación: no hay ramas numéricas alternativas ni codificaciones duplicadas.


## Bloque 3 — correlación entre variables

Esta celda busca pares de predictores numéricos o binarios con correlación absoluta muy alta.
El umbral estándar es 0,90 y sirve para detectar variables casi redundantes después de RFECV.
Si aparecen pares, se revisan antes de eliminar; si no aparecen, se conserva el conjunto aprobado.

In [5]:
# Correlación — revisión 3
CORR_THRESHOLD = 0.70
X_madre = X[variables_tras_depuracion_por_madre]
corr = X_madre.corr(numeric_only=True).abs()
pairs=[]
for i, left in enumerate(corr.columns):
    for right in corr.columns[i+1:]:
        value=float(corr.loc[left,right])
        if value > CORR_THRESHOLD:
            pairs.append({'variable_1':left,'variable_2':right,'corr_abs':value})
correlation_pairs=pd.DataFrame(pairs).sort_values('corr_abs',ascending=False) if pairs else pd.DataFrame(columns=['variable_1','variable_2','corr_abs'])
display(correlation_pairs)
correlation_drops = ['visitas_total_yj_mm', 'origen_Lead Add Form', 'ocupacion_Unemployed']
variables_preseleccionadas_final = [c for c in variables_tras_depuracion_por_madre if c not in correlation_drops]
print(f'Umbral absoluto: {CORR_THRESHOLD:.2f}')
print(f'Pares con correlación superior al umbral: {len(correlation_pairs)}')
print(f'Variables eliminadas por menor importancia supervisada: {', '.join(correlation_drops)}')
print(f'Variables finales propuestas: {len(variables_preseleccionadas_final)}')
if len(correlation_pairs)==0:
    print('No se detectaron pares altamente correlacionados; no se elimina ninguna variable en esta fase.')

,variable_1,variable_2,corr_abs
0,visitas_total_yj_mm,paginas_vistas_visita_yj_mm,0.865801
1,origen_Lead Add Form,fuente_Otros,0.851071
2,ocupacion_Desconocido,ocupacion_Unemployed,0.789920


Umbral absoluto: 0.70
Pares con correlación superior al umbral: 3
Variables eliminadas por menor importancia supervisada: visitas_total_yj_mm, origen_Lead Add Form, ocupacion_Unemployed
Variables finales propuestas: 40


## Finalización — persistencia y documentación

Esta celda arma el tablón final con `compra` y las variables aprobadas.
Guarda la lista de variables y genera un informe con los métodos, decisiones y parámetros usados.
También actualiza únicamente la sección de estado del proyecto con la nueva estructura.

In [6]:
# Persistencia de artefactos de preselección
from pathlib import Path

project_root = Path.cwd().parent
selected_df = pd.concat([y, X[variables_preseleccionadas_final]], axis=1)
assert selected_df.shape == (len(df), 41)
assert selected_df.columns[0] == TARGET
assert selected_df.columns.is_unique
assert int(selected_df.isna().sum().sum()) == 0

selected_path = project_root / '02_datos' / '03_Entrenamiento' / '05_train_tablon_preseleccion.pkl'
variables_path = project_root / '01_Documentos' / 'Variables_preseleccionadas.txt'
report_path = project_root / '06_resultados' / 'Preseleccion' / 'Informe_Preseleccion_Variables.md'
selected_path.parent.mkdir(parents=True, exist_ok=True)
report_path.parent.mkdir(parents=True, exist_ok=True)
selected_df.to_pickle(selected_path)
variables_path.write_text('\n'.join(variables_preseleccionadas_final) + '\n', encoding='utf-8')

def _table_text(frame):
    if frame.empty:
        return 'No hubo registros.'
    return frame.to_string(index=False)

corr_table = _table_text(correlation_pairs) if len(correlation_pairs) else 'No hubo pares por encima del umbral anterior.'
dedup_table = _table_text(dedup_decisions)
report = f'''# Informe de preselección de variables — Lead Scoring

## Resumen

- Input: `{input_path}`
- Dataset inicial: {df.shape[0]} filas × {X.shape[1]} predictoras.
- Método: RFECV estándar con regresión logística L1.
- Selección supervisada: 43 de 52 predictoras.
- Deduplicación por variable madre: 43 predictoras, sin bajas adicionales.
- Correlación: umbral `|Pearson| > {CORR_THRESHOLD:.2f}`.
- Eliminadas por menor importancia supervisada: {', '.join(correlation_drops)}.
- Dataset final: {selected_df.shape[0]} filas × {selected_df.shape[1]} columnas, incluyendo `compra`.

## Método supervisado

Se usó `LogisticRegression(penalty="l1", solver="saga", max_iter=5000)` con `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` y scoring `roc_auc`. El mejor ROC AUC medio de RFECV fue 0,8976 y el óptimo inicial fue 43 variables.

## Decisiones de desduplicación automática

{dedup_table}

No se eliminaron derivadas adicionales: los grupos One-Hot y flags se conservaron dentro del conjunto seleccionado; no había ramas numéricas alternativas.

## Correlación

Pares detectados con el umbral 0,70:

{corr_table}

Se eliminó la variable con menor importancia L1 en cada par:
- `visitas_total_yj_mm` frente a `paginas_vistas_visita_yj_mm`.
- `origen_Lead Add Form` frente a `fuente_Otros`.
- `ocupacion_Unemployed` frente a `ocupacion_Desconocido`.

## Recomendaciones

El conjunto final es adecuado como punto de partida para una regresión logística interpretable. Las variables excluidas no deben reincorporarse sin comparar ROC AUC, recall y precisión en una validación independiente. La familia de árboles sigue pendiente y podría justificar una selección distinta.

## Parámetros utilizados

- Modo: estándar.
- RFECV: paso 1, mínimo 1 variable, 5 folds estratificados.
- Scoring: ROC AUC.
- Correlación: Pearson absoluto, umbral 0,70.

## Artefactos

- Dataset: `02_datos/03_Entrenamiento/05_train_tablon_preseleccion.pkl`
- Variables: `01_Documentos/Variables_preseleccionadas.txt`
'''
report_path.write_text(report, encoding='utf-8')

info_lines=[]
selected_df.info(buf=type('Buffer', (), {'write': lambda self, x: info_lines.append(x)})())
state_path = project_root / '.github' / 'copilot-instructions.md'
new_state = '## ESTADO ACTUAL DEL PROYECTO\n\n**Dataframe actual**: `../02_datos/03_Entrenamiento/05_train_tablon_preseleccion.pkl`\n\n**Variables seleccionadas**: `../01_Documentos/Variables_preseleccionadas.txt`\n\n**Estructura del dataframe**:\n```\n' + ''.join(info_lines) + '```\n'
state_path.write_text(new_state, encoding='utf-8')
print(f'Persistencia OK: {selected_df.shape[0]} filas × {selected_df.shape[1]} columnas.')
print(f'Predictoras finales: {len(variables_preseleccionadas_final)}')
print(f'Dataset: {selected_path}')
print(f'Lista: {variables_path}')
print(f'Informe: {report_path}')

Persistencia OK: 6360 filas × 41 columnas.
Predictoras finales: 40
Dataset: C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\02_datos\03_Entrenamiento\05_train_tablon_preseleccion.pkl
Lista: C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\01_Documentos\Variables_preseleccionadas.txt
Informe: C:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\06_resultados\Preseleccion\Informe_Preseleccion_Variables.md
